# Xử lý giá trị thiếu và ngoại lai

**Phạm vi notebook:** xử lý dữ liệu thiếu và kiểm tra ngoại lai cho `gia`, `dien_tich`; lưu một bản dữ liệu cleaned riêng. Notebook không tạo đặc trưng, nhãn giá/m² hay file model-ready để không trùng nhiệm vụ Feature Engineering và đóng gói báo cáo.

**Quyết định:** giá/diện tích thiếu được điền median theo `vi_tri`, thiếu median nhóm thì dùng median toàn bộ; giữ giá trị gốc và cờ thiếu để truy vết. `so_phong_ngu` được giữ thiếu vì tin không nhắc đến số phòng không có nghĩa là số phòng bằng 0. Thiếu giá/diện tích có thể là **MNAR** do người bán không công khai; khác biệt thiếu giữa khu vực có thể gợi ý **MAR**. Không thể kết luận cơ chế chỉ từ dữ liệu quan sát.

Ngoại lai được đánh dấu bằng IQR (1,5 × IQR) và đối chiếu với tiêu đề tin. Không tự động xóa/cắt các cực trị: thống kê mô tả ở notebook 03 cho thấy đuôi giá/diện tích rất rộng, còn tin gốc có thể là quỹ đất/dự án hợp lệ. Giữ các giá trị này và cờ ngoại lai để tránh làm mất tín hiệu thị trường; việc cắt ngọn hoặc phân tích độ nhạy nên được quyết định sau khi rà soát tin cụ thể.

## 1. Đọc và kiểm tra dữ liệu

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Tìm Project để notebook chạy được từ thư mục Project hoặc notebooks.
here = Path.cwd().resolve()
data_relative_path = Path('data') / 'processed' / 'cafeland_cleaned.csv'
project = next(
    (folder for folder in [here, *here.parents]
     if (folder / data_relative_path).is_file()),
    None,
)
if project is None:
    raise FileNotFoundError(f'Không tìm thấy {data_relative_path} từ {here}')

df_source = pd.read_csv(project / data_relative_path, encoding='utf-8-sig')
df_source.columns = df_source.columns.str.strip().str.lower()
required_columns = {'gia', 'dien_tich', 'vi_tri', 'so_phong_ngu'}
if not required_columns.issubset(df_source.columns):
    raise ValueError(f'Thiếu cột bắt buộc: {sorted(required_columns - set(df_source.columns))}')

for column in ['gia', 'dien_tich', 'so_phong_ngu']:
    df_source[column] = pd.to_numeric(df_source[column], errors='coerce')
df_source = df_source.replace(r'^\s*$', np.nan, regex=True)

print(f'Đã đọc {len(df_source):,} dòng, {len(df_source.columns)} cột.')
display(pd.DataFrame({
    'Số lượng thiếu': df_source.isna().sum(),
    'Tỷ lệ thiếu (%)': (df_source.isna().mean() * 100).round(2),
}).sort_values('Tỷ lệ thiếu (%)', ascending=False))
display(df_source[['gia', 'dien_tich', 'so_phong_ngu']].describe().T)

Đã đọc 6,500 dòng, 9 cột.


,Số lượng thiếu,Tỷ lệ thiếu (%)
so_phong_ngu,3747,57.65
dien_tich,1782,27.42
gia,492,7.57
tieu_de,0,0.00
vi_tri,0,0.00
ngay_dang,0,0.00
link_nguon,0,0.00
mo_ta,0,0.00
nguoi_ban,0,0.00


,count,mean,std,min,25%,50%,75%,max
gia,6008.0,3.198635e+10,2.071517e+11,1.48,3.741500e+09,7.663000e+09,1.850000e+10,7.140000e+12
dien_tich,4718.0,4.829983e+03,2.638654e+05,2.00,6.000000e+01,9.000000e+01,1.760000e+02,1.809000e+07
so_phong_ngu,2753.0,4.192154e+00,7.780635e+00,1.00,2.000000e+00,2.000000e+00,4.000000e+00,1.060000e+02


## 2. So sánh tỷ lệ thiếu theo khu vực

Nếu tỷ lệ thiếu thay đổi giữa các khu vực, việc thiếu có thể liên quan đến thông tin quan sát được (gợi ý MAR). Đây chỉ là dấu hiệu tham khảo, không chứng minh được MAR hay loại trừ MNAR.

In [2]:
missing_by_area = df_source.assign(
    gia_thieu=df_source['gia'].isna(),
    dien_tich_thieu=df_source['dien_tich'].isna(),
).groupby('vi_tri', dropna=False).agg(
    so_tin=('gia_thieu', 'size'),
    gia_thieu_pct=('gia_thieu', 'mean'),
    dien_tich_thieu_pct=('dien_tich_thieu', 'mean'),
)
missing_by_area = missing_by_area[missing_by_area['so_tin'] >= 20]
missing_by_area[['gia_thieu_pct', 'dien_tich_thieu_pct']] *= 100
display(missing_by_area.sort_values('so_tin', ascending=False).head(10).round(1))
for column in ['gia_thieu_pct', 'dien_tich_thieu_pct']:
    rates = missing_by_area[column]
    print(f'{column}: {rates.min():.1f}%–{rates.max():.1f}% giữa các khu vực '
          f'(chỉ tính nhóm có ít nhất 20 tin).')

,so_tin,gia_thieu_pct,dien_tich_thieu_pct
vi_tri,,,
"An Khánh, Thành phố Hồ Chí Minh",383,4.7,53.0
"Bình Dương, Thành phố Hồ Chí Minh",337,68.2,89.3
"Lái Thiêu, Thành phố Hồ Chí Minh",301,0.3,90.7
"Tân Hưng, Thành phố Hồ Chí Minh",291,1.7,1.4
"Dĩ An, Thành phố Hồ Chí Minh",185,5.4,44.9
"Cần Giờ, Thành phố Hồ Chí Minh",151,8.6,46.4
"Bàn Cờ, Thành phố Hồ Chí Minh",145,1.4,3.4
"Hiệp Bình, Thành phố Hồ Chí Minh",135,33.3,29.6
"Bình Trị Đông, Thành phố Hồ Chí Minh",131,0.0,0.8


gia_thieu_pct: 0.0%–68.2% giữa các khu vực (chỉ tính nhóm có ít nhất 20 tin).
dien_tich_thieu_pct: 0.0%–90.7% giữa các khu vực (chỉ tính nhóm có ít nhất 20 tin).


## 3. Đánh dấu và rà soát ngoại lai bằng IQR

IQR đánh dấu giá trị cần kiểm tra, không tự nó chứng minh giá trị sai. Xem tiêu đề và đường dẫn tin gốc trước khi quyết định loại/cắt; các tin dự án lớn có thể hợp lệ.

In [3]:
bounds = {}
outlier_report = []

for column in ['gia', 'dien_tich']:
    observed = df_source.loc[df_source[column] > 0, column]
    if observed.empty:
        raise ValueError(f'Không có giá trị dương để tính IQR cho {column}.')

    q1 = observed.quantile(0.25)
    q3 = observed.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    bounds[column] = (lower, upper)

    outlier = df_source[column].notna() & (
        (df_source[column] < lower) | (df_source[column] > upper)
    )
    outlier_report.append({
        'Biến': column,
        'Q1': q1,
        'Q3': q3,
        'Ngưỡng dưới': lower,
        'Ngưỡng trên': upper,
        'Số ngoại lai': int(outlier.sum()),
        'Tỷ lệ ngoại lai (%)': round(outlier.mean() * 100, 2),
    })

display(pd.DataFrame(outlier_report).set_index('Biến').round(2))

# Xem các tin có giá/diện tích lớn nhất để đặt ngoại lai vào ngữ cảnh thực tế.
for column in ['gia', 'dien_tich']:
    print(f'Các tin có {column} lớn nhất:')
    display(df_source.nlargest(5, column)[
        ['tieu_de', column, 'vi_tri', 'link_nguon']
    ])

,Q1,Q3,Ngưỡng dưới,Ngưỡng trên,Số ngoại lai,Tỷ lệ ngoại lai (%)
Biến,,,,,,
gia,3.741500e+09,1.850000e+10,-1.839625e+10,4.063775e+10,711,10.94
dien_tich,6.000000e+01,1.760000e+02,-1.140000e+02,3.500000e+02,599,9.22


Các tin có gia lớn nhất:


,tieu_de,gia,vi_tri,link_nguon
4389,"Quỹ Đất PTDA 5.1ha Trần Não & Mai Chí Thọ, Q2 ...",7.140000e+12,"An Khánh, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-51ha-t...
4346,"Quỹ Đất PTDA 1,809ha KĐT & KCN Tân Lâm, X. Hòa...",6.512000e+12,"Hòa Hội, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-1809ha...
4272,Chuyển nhượng nhà hẻm xe hơi 8m đường Hồng Bàn...,5.790000e+12,"Bình Thới, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/chuyen-nhuong-nha-h...
4457,"Quỹ Đất PTDA (70ha) MT Trần Đại Nghĩa, Bình Ch...",5.600000e+12,"Bình Lợi, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-70ha-m...
3424,"Quỹ Đất PTDA 36ha MT Nguyễn Hữu Thọ, Phước Kiể...",5.400000e+12,"Hiệp Phước, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-36ha-m...


Các tin có dien_tich lớn nhất:


,tieu_de,dien_tich,vi_tri,link_nguon
4346,"Quỹ Đất PTDA 1,809ha KĐT & KCN Tân Lâm, X. Hòa...",18090000.0,"Hòa Hội, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-1809ha...
4457,"Quỹ Đất PTDA (70ha) MT Trần Đại Nghĩa, Bình Ch...",700000.0,"Bình Lợi, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-70ha-m...
4424,"M&A 2 Cảng Mỹ Xuân, BRVT (P. Phú Mỹ HCM)",680000.0,"Phú Mỹ, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/ma-2-cang-my-xuan-b...
3424,"Quỹ Đất PTDA 36ha MT Nguyễn Hữu Thọ, Phước Kiể...",360000.0,"Hiệp Phước, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/quy-dat-ptda-36ha-m...
6345,CHUYỂN NHƯỢNG ĐẤT LÀM DỰ ÁN BỆNH VIỆN QUY MÔ 1...,290000.0,"An Khánh, Thành phố Hồ Chí Minh",https://nhadat.cafeland.vn/chuyen-nhuong-dat-l...


## 4. Điền thiếu, giữ dấu vết và lưu bản cleaned

Không xóa dòng ngoại lai: chỉ thêm cờ IQR để người phân tích có thể lọc hoặc kiểm tra độ nhạy sau khi xác minh tin. Giá/diện tích không dương được xem là không hợp lệ và chuyển thành thiếu. Điền giá/diện tích bằng median theo khu vực; nếu nhóm không có giá trị quan sát thì dùng median toàn bộ. Giữ cột gốc và cờ để phân biệt dữ liệu thật với dữ liệu được điền. Số phòng ngủ vẫn để thiếu.

In [4]:
df_clean = df_source.copy()

for column in ['gia', 'dien_tich']:
    lower, upper = bounds[column]
    original = df_source[column]
    invalid = original.notna() & original.le(0)

    # Lưu giá trị gốc và cờ trước khi điền.
    df_clean[f'{column}_goc'] = original
    df_clean[f'{column}_thieu'] = original.isna() | invalid
    df_clean[f'{column}_ngoai_lai'] = original.notna() & (
        (original < lower) | (original > upper)
    )

    # Giá/diện tích <= 0 không hợp lệ; không xóa ngoại lai có thể là tin thật.
    df_clean.loc[invalid, column] = np.nan
    median_by_area = df_clean.groupby('vi_tri', dropna=False)[column].transform('median')
    df_clean[column] = df_clean[column].fillna(median_by_area).fillna(df_clean[column].median())

# Không đoán số phòng ngủ; cờ cho biết trường này vốn bị thiếu.
df_clean['so_phong_ngu_thieu'] = df_source['so_phong_ngu'].isna()

assert len(df_clean) == len(df_source), 'Không được mất dòng khi làm sạch.'
assert df_clean[['gia', 'dien_tich']].notna().all().all(), 'Giá/diện tích vẫn còn thiếu.'

output_file = project / 'data' / 'processed' / 'cafeland_missing_outliers_cleaned.csv'
df_clean.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f'Đã lưu {len(df_clean):,} dòng vào: {output_file}')
print('Đã kiểm tra: giữ nguyên số dòng; giá và diện tích sau xử lý không còn thiếu.')

Đã lưu 6,500 dòng vào: D:\Study\LapTrinhPhanTichDuLieu\Project\data\processed\cafeland_missing_outliers_cleaned.csv
Đã kiểm tra: giữ nguyên số dòng; giá và diện tích sau xử lý không còn thiếu.


## Kết luận và bàn giao

- Bản `cafeland_missing_outliers_cleaned.csv` là đầu ra duy nhất của notebook này. File nguồn không bị ghi đè.
- Giá/diện tích được điền có cờ thiếu và giá trị gốc; ngoại lai được đánh dấu, không bị xóa hoặc cắt tự động vì có thể đại diện cho bất động sản/dự án lớn hợp lệ.
- Notebook này **không** tạo `price_per_m2`, mã hóa/phân nhóm đặc trưng hoặc xuất `cafeland_model_ready.csv`; các bước đó thuộc nhiệm vụ Feature Engineering và đóng gói của thành viên khác. Khi dùng dữ liệu để huấn luyện mô hình, cần tính ngưỡng và median từ tập train, không dùng giá trị đã điền làm nhãn.